In [ ]:
%pip install anthropic python-dotenv pandas --upgrade --quiet

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

In [3]:
# import Lenny Newsletter - Sheet1.csv
import pandas as pd

df = pd.read_csv("Lenny Newsletter - Sheet1.csv")
df.head()

,Title,Link,Description,Published,Author,Hearts,Comments
0,How Duolingo reignited user growth,https://www.lennysnewsletter.com/p/how-duoling...,The story behind Duolingo's 350% growth accele...,"Feb 28, 2023",JORGE MAZAL,673.0,59.0
1,How the biggest consumer apps got their first ...,https://www.lennysnewsletter.com/p/how-the-big...,Considering every startup confronts this quest...,"May 12, 2020",LENNY RACHITSKY,452.0,29.0
2,You should be playing with GPTs at work,https://www.lennysnewsletter.com/p/you-should-...,20 examples of how people are using custom GPT...,Feb 20,LENNY RACHITSKY,354.0,21.0
3,How to use ChatGPT in your PM work,https://www.lennysnewsletter.com/p/how-to-use-...,Real-life examples (and actual prompts) of how...,NaN,•,362.0,1.0
4,The 10 commandments of salary negotiation,https://www.lennysnewsletter.com/p/negotiating...,"Guest post by Niya Dragova, co-founder of Candor","Sep 24, 2021",LENNY RACHITSKY,157.0,53.0


In [4]:
# urls = list of "Title" and "Link" as a list of tuples
urls = list(zip(df["Title"], df["Link"]))

print(urls[:5])

[('How Duolingo reignited user growth', 'https://www.lennysnewsletter.com/p/how-duolingo-reignited-user-growth'), ('How the biggest consumer apps got their first 1,000 users', 'https://www.lennysnewsletter.com/p/how-the-biggest-consumer-apps-got'), ('You should be playing with GPTs at work', 'https://www.lennysnewsletter.com/p/you-should-be-playing-with-gpts-at'), ('How to use ChatGPT in your PM work', 'https://www.lennysnewsletter.com/p/how-to-use-chatgpt-in-your-pm-work'), ('The 10 commandments of salary negotiation', 'https://www.lennysnewsletter.com/p/negotiating-comp')]


In [ ]:
from anthropic import Anthropic

client = Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1000,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Hello"
                }
            ]
        }
    ]
)
print(message.content[0].text)

In [ ]:
# loop through the files in the folder and summarize each
summaries = {}

if os.path.isdir("lenny"):
    # get a list of urls
    for url in urls[:5]:
        with open(f"lenny/{url[1].split('/')[-1]}.txt", "r") as file:
            text = file.read()

        message = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1000,
            temperature=0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"<text>{text}</text>\nPull out one nonobvious insight from this text. Use quotes where relevant. Be extremely concise"
                        }
                    ]
                }
            ]
        )
        summaries[url] = message.content[0].text

    # print the summaries
    for url, summary in summaries.items():
        print(url)
        print(summary)
        print("\n")
else:
    print("Skipping individual article summaries — 'lenny/' folder not found. Run scrape.ipynb first to generate individual article files.")

In [ ]:
if not os.path.exists("all_text.txt"):
    # loop through the files in the folder and add all the text together in one file, separated by a divider
    all_text = ""
    for url in urls[:50]:
        with open(f"lenny/{url[1].split('/')[-1]}.txt", "r") as file:
            text = file.read()
            all_text += "<article>" + text + "</article>\n------------------------------------\n"

    # how many tokens based on length of the text. 1 token is 4 characters
    print(len(all_text) / 4)

    with open("all_text.txt", "w") as file:
        file.write(all_text)
else:
    print("all_text.txt already exists, skipping generation.")

In [ ]:
# summarize the combined text
with open("all_text.txt", "r") as file:
    text = file.read()

message = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1000,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{text}\nSummarize these articles, analyze the word choice, identify any patterns, estimate the reading level, and tell me any other non obvious insights about this publication."
                }
            ]
        }
    ]
)

print(message.content[0].text)

In [ ]:
titles = "\n".join(df["Title"].tolist())

prompt_template = f"""Successful titles follow templates. Here are examples of templates:

Template: "The Blank of Blank"
Description: The format is where the first blank is a word not usually associated with the topic, and the second blank is the topic of the content.
Example 1 - The War of Art
Example 2 - The Psychology of Money
Example 3 - The Subtle Art of Not Giving a Fuck

Template: "Odd Topic"
Description: The first word is an odd adjective to pair with the topic of the content, and the second word is the topic of the content.
Example 1 - Atomic Habits
Example 2 - Extreme Ownership
Example 3 - Deep Work

Template: "The Definitive Guide"
Description: The first word is an unusual adjective not normally associated with the topic of the content, and the second word is the topic of the content.
Example 1 - WebAssembly: The Definitive Guide
Example 2 - Cassandra: The Definitive Guide
Example 3 - Kafka: The Definitive Guide, 2nd Edition

Categorize the following titles into templates. Make up new templates based on what you find. Titles can be assigned to multiple templates. Only list a maximum of 3 examples of titles that fit each template, but skip the template if there are not at least 2 examples. Ignore edition numbers or versions in your analysis. Do not fit a title to a template if it doesn't apply. The example templates may not be suitable for these titles.

Titles:
{ titles }
"""

message = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=1000,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt_template
                }
            ]
        }
    ]
)

print(message.content[0].text)